# MeTRAbs Colab Server (mirrors teammate's + intrinsics JSON support)

**IMPORTANT:** Runtime > Change runtime type > GPU (T4) BEFORE running.

In [ ]:
# Cell 1: Install deps
!pip install -q tensorflow opencv-python-headless certifi fastapi uvicorn pyngrok python-multipart nest-asyncio

In [ ]:
# Cell 2: GPU verify
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU ready!')
else:
    print('WARNING: No GPU - go to Runtime > Change runtime type > GPU')

In [ ]:
# Cell 3: Download + load MeTRAbs model
import os, ssl, certifi
ssl_context = ssl.create_default_context(cafile=certifi.where())
ssl._create_default_https_context = lambda: ssl_context

MODEL_TYPE = 'metrabs_mob3l_y4t'
SKELETON = 'smpl_24'
CACHE_DIR = './metrabs_models'
os.makedirs(CACHE_DIR, exist_ok=True)

fname = f'{MODEL_TYPE}_20211019.zip'
origin = f'https://omnomnom.vision.rwth-aachen.de/data/metrabs/{fname}'

print('Downloading MeTRAbs model...')
model_zip = tf.keras.utils.get_file(fname=fname, origin=origin, extract=True, cache_subdir='.', cache_dir=CACHE_DIR)

model_path = None
for root, dirs, fnames in os.walk(os.path.dirname(model_zip)):
    if 'saved_model.pb' in fnames:
        model_path = root; break

print(f'Loading model from: {model_path}')
model = tf.saved_model.load(model_path)
edges = model.per_skeleton_joint_edges[SKELETON].numpy().tolist()
print(f'Model loaded! Skeleton: {SKELETON}, edges: {len(edges)}')

In [ ]:
# Cell 4: ngrok auth token
# Sign up free at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""  # <-- PASTE YOUR NGROK AUTHTOKEN HERE

from pyngrok import ngrok
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    print('ngrok auth set')
else:
    print('WARNING: No ngrok token set — tunnel may rate-limit')

In [ ]:
# Cell 5: FastAPI server mirroring teammate's main.py + run.py + intrinsics support
import json, csv, shutil, subprocess, os
import numpy as np
import cv2
from pathlib import Path
from time import time
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio, uvicorn

nest_asyncio.apply()

BASE_DIR = Path('/content/metrabs-server')
UPLOAD_FOLDER = BASE_DIR / 'uploads'
OUTPUT_FOLDER = BASE_DIR / 'outputs'
UPLOAD_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
app.mount('/outputs', StaticFiles(directory=str(OUTPUT_FOLDER)), name='outputs')

RESIZE_FACTOR = 0.25   # matches teammate's run.py
FRAME_SKIP = 2         # matches teammate's run.py

def delete_all_files_in_folder(folder):
    for f in folder.iterdir():
        if f.is_file():
            try: f.unlink()
            except Exception: pass

def draw_skeletons(frame, poses2d, edges, w, h):
    if poses2d is None: return
    for pose in poses2d:
        pts = pose.astype(int)
        for i, j in edges:
            if i < len(pts) and j < len(pts):
                x1, y1 = pts[i]; x2, y2 = pts[j]
                if 0<=x1<w and 0<=y1<h and 0<=x2<w and 0<=y2<h:
                    cv2.line(frame, (x1,y1), (x2,y2), (0,200,255), 2)
        for x, y in pts:
            if 0<=x<w and 0<=y<h:
                cv2.circle(frame, (x,y), 2, (0,128,255), -1)

def process_video(input_path, video_out, csv_out, intrinsic_matrix=None):
    cap = cv2.VideoCapture(str(input_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_w = int(width * RESIZE_FACTOR)
    out_h = int(height * RESIZE_FACTOR)
    out_fps = fps / FRAME_SKIP if FRAME_SKIP > 0 else fps

    intrinsics_tensor = None
    if intrinsic_matrix is not None:
        scaled = intrinsic_matrix.copy()
        scaled[0, :] *= RESIZE_FACTOR
        scaled[1, :] *= RESIZE_FACTOR
        intrinsics_tensor = tf.constant(scaled, dtype=tf.float32)
        print(f'Using intrinsics (scaled):\n{scaled}', flush=True)

    temp_out = str(OUTPUT_FOLDER / 'temp.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_out, fourcc, out_fps, (out_w, out_h))

    with open(csv_out, 'w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['frame', 'person_id', 'joint_id', 'x', 'y', 'z'])

        source_idx = 0; processed = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            source_idx += 1
            if FRAME_SKIP > 1 and source_idx % FRAME_SKIP != 0: continue
            processed += 1

            frame_small = cv2.resize(frame, (out_w, out_h))
            rgb = cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB)

            with tf.device('/GPU:0'):
                image_tensor = tf.convert_to_tensor(rgb)
                if intrinsics_tensor is not None:
                    pred = model.detect_poses(image_tensor, skeleton=SKELETON, intrinsic_matrix=intrinsics_tensor)
                else:
                    pred = model.detect_poses(image_tensor, skeleton=SKELETON)

            poses2d = pred.get('poses2d')
            poses3d = pred.get('poses3d')
            if hasattr(poses2d, 'numpy'): poses2d = poses2d.numpy()
            if hasattr(poses3d, 'numpy'): poses3d = poses3d.numpy()

            if poses3d is not None:
                for pid, pose in enumerate(poses3d):
                    for jid, (x,y,z) in enumerate(pose):
                        writer.writerow([processed, pid, jid, float(x), float(y), float(z)])

            draw_skeletons(frame_small, poses2d, edges, out_w, out_h)
            out.write(frame_small)

            if processed % 25 == 0:
                print(f'Processed {processed} frames', flush=True)

    cap.release(); out.release()

    # Compress with ffmpeg
    subprocess.run(['ffmpeg','-i',temp_out,'-vcodec','libx264','-crf','30','-preset','fast','-y', str(video_out)], check=True)
    if os.path.exists(temp_out): os.remove(temp_out)

@app.get('/')
def home():
    return {'message': 'MeTRAbs Colab server running', 'gpu': len(gpus) > 0}

@app.post('/upload')
async def upload_video(
    file: UploadFile = File(...),
    intrinsics_json: str = Form(None),
    focal_length_x: str = Form(None),
    focal_length_y: str = Form(None),
    principal_point_x: str = Form(None),
    principal_point_y: str = Form(None),
):
    print('UPLOAD REQUEST RECEIVED', flush=True)
    print(f'filename: {file.filename}', flush=True)
    if not file.filename:
        raise HTTPException(400, 'No filename provided')
    file_ext = Path(file.filename).suffix.lower()
    if file_ext != '.mp4':
        raise HTTPException(400, f'Only .mp4 files allowed. Received: {file.filename}')

    delete_all_files_in_folder(UPLOAD_FOLDER)
    delete_all_files_in_folder(OUTPUT_FOLDER)

    input_path = UPLOAD_FOLDER / file.filename
    output_video = OUTPUT_FOLDER / 'output_video_1.mp4'
    output_csv = OUTPUT_FOLDER / 'output_file_1.csv'

    with open(input_path, 'wb') as f:
        shutil.copyfileobj(file.file, f)
    print(f'Saved upload: {input_path}', flush=True)

    intrinsic_np = None
    if intrinsics_json:
        try:
            data = json.loads(intrinsics_json)
            matrix = data.get('camera_intrinsics', {}).get('intrinsic_matrix')
            if matrix:
                intrinsic_np = np.array(matrix, dtype=np.float32)
                print(f'Intrinsics loaded: fx={intrinsic_np[0,0]:.1f}, fy={intrinsic_np[1,1]:.1f}', flush=True)
        except Exception as e:
            print(f'Intrinsics JSON parse error: {e}', flush=True)
    elif focal_length_x:
        fx = float(focal_length_x); fy = float(focal_length_y or fx)
        cx = float(principal_point_x or 0); cy = float(principal_point_y or 0)
        intrinsic_np = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]], dtype=np.float32)
        print(f'Intrinsics from fields: fx={fx}, fy={fy}', flush=True)

    try:
        t0 = time()
        process_video(input_path, output_video, output_csv, intrinsic_np)
        elapsed = time() - t0
        print(f'Done in {elapsed:.1f}s', flush=True)
    except Exception as e:
        raise HTTPException(500, f'Processing failed: {e}')
    finally:
        file.file.close()

    if not output_video.exists():
        raise HTTPException(500, f'Output video not created: {output_video}')
    if not output_csv.exists():
        raise HTTPException(500, f'CSV not created: {output_csv}')

    return {
        'message': 'Video processed successfully',
        'input_file': input_path.name,
        'output_file': output_video.name,
        'output_url': f'/outputs/{output_video.name}',
        'csv_file': output_csv.name,
        'csv_url': f'/outputs/{output_csv.name}',
        'used_intrinsics': intrinsic_np is not None,
        'processing_time_seconds': round(elapsed, 1),
    }

print('Server defined')

In [ ]:
# Cell 6: Start ngrok tunnel + uvicorn
public_url = ngrok.connect(8000)
print('=' * 60)
print(f'YOUR SERVER URL: {public_url}')
print('=' * 60)
print('Paste the https://...ngrok-free.app part into the app')
uvicorn.run(app, host='0.0.0.0', port=8000)